# Analysis of the training runs

Reads every run saved by `train.py` in `results/`, trained with either sampler (Metropolis, folder suffix `_m`, or Gibbs, `_g`), and answers:

1. **Did each run converge?** Final energy against the exact $E_0 = N\,d\,\omega/2$, and $\mathrm{Var}(E_{loc}) \to 0$.
2. **How did training go?** Energy, variance, acceptance and $\alpha$ per SR iteration.
3. **Does the acceptance drift matter?** Acceptance during training against convergence speed.
4. **Is the trained wavefunction right?** A cut of $\log\psi$ compared with the exact ground state.
5. **What is the proper energy?** A long sampling run at fixed $\theta$, with an error bar from independent chains.
6. **Is the phase constant?** The ground state is real and positive, so the learned phase must be the same for every configuration.

Run it from VS Code with the `.venv` kernel. Choose which runs to analyse in **section 1b**; section 5 and the first part of section 6 analyse one run, chosen in `RUN_NAME`.

In [ ]:
import os
#Do not let JAX take 75% of the GPU memory, so the notebook can run next to a training
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

import sys
import tomllib
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp

#The notebook lives in notebooks/, but src/ and results/ are in the project root (where pyproject.toml is)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

from src.boson_trap import boson_trap
from src.nqs import NQS
from src.sampler_m import MetroSampler
from src.sampler_g import GibbsSampler
from src.sr_optimizer import flatten_params

#Folders with run outputs (train.py's output_dir). Add more to compare runs saved elsewhere
RESULTS_DIRS = [ROOT / "results"]
ACC_MIN, ACC_MAX = 0.25, 0.4   #same acceptance range that train.py warns about at the end of training
SAMPLERS = {"metropolis": MetroSampler, "gibbs": GibbsSampler}
SAMPLER_TAGS = {"metropolis": "m", "gibbs": "g"}   #same tags as the run folder names
print("Project root:", ROOT, "| JAX devices:", jax.devices())

## 1. Load the runs

`load_run` reads one run folder and computes the same summary that `train.py` prints (mean of the last 10% of the iterations). It also computes the first iteration where the relative error drops below `REL_TOL`, to measure how fast each run converges, and marks as **diverged** the runs whose history or parameters contain NaN or inf.

`sampler_config` reads which sampler a run used. Runs saved before the Gibbs sampler have a flat `[sampler]` section and always used Metropolis; newer ones have `[sampler] type` and one subsection per sampler.

All runs of `RESULTS_DIRS` are loaded into `all_runs`, sorted by date (the timestamp in the folder name), and listed with an index.

In [ ]:
REL_TOL = 1e-3   #threshold used to measure convergence speed

def sampler_config(cfg):
    #New configs: [sampler] type + [sampler.metropolis] / [sampler.gibbs]. Old configs: a flat [sampler]
    #section, always Metropolis. Returns the sampler type and the keys of its section
    s = cfg["sampler"]
    if "type" in s:
        return s["type"], s[s["type"]]
    return "metropolis", s

def load_run(run_dir):
    #Each run has its own config, so runs with different N, dim or hyperparameters can be compared
    with open(run_dir / "config.toml", "rb") as f:
        cfg = tomllib.load(f)
    data = np.load(run_dir / "results.npz")

    system = boson_trap(**cfg["system"])
    sampler_type, sampler_cfg = sampler_config(cfg)
    energy, variance = data["energy"], data["variance"]
    acceptance, alpha = data["acceptance"], data["alpha"]
    rel_err = np.abs(energy - system.exact_energy) / system.exact_energy

    #Same estimate as train.py: mean of the last 10% of the iterations (error bar ignores correlations)
    n_last = max(1, len(energy) // 10)
    below = np.nonzero(rel_err < REL_TOL)[0]

    return {
        "name": run_dir.name, "dir": run_dir, "cfg": cfg, "system": system,
        "sampler_type": sampler_type, "sampler_cfg": sampler_cfg,
        "theta": data["theta"], "energy": energy, "variance": variance,
        "acceptance": acceptance, "alpha": alpha, "rel_err": rel_err,
        #std of the phase over the samples of every iteration; runs saved before it was added do not have it
        "phase_std": data["phase_std"] if "phase_std" in data.files else None,
        "E_final": energy[-n_last:].mean(),
        "E_err": energy[-n_last:].std() / np.sqrt(n_last),
        "acc_last": acceptance[-n_last:].mean(),
        "iter_tol": int(below[0]) + 1 if below.size else None,   #None: never reached REL_TOL
        #A diverged run (NaN/inf) cannot be rebuilt or sampled in sections 4-5
        "diverged": not (np.all(np.isfinite(energy)) and np.all(np.isfinite(data["theta"]))),
        #Folder names are N{N}_{YYYYmmdd-HHMMSS[-ffffff]}[_m|_g]: the second field sorts by date
        "time": run_dir.name.split("_")[1],
        "label": (f"N={system.n_particles} d={system.dim} {SAMPLER_TAGS[sampler_type]} δ={sampler_cfg['step_size']} "
                  f"η={cfg['sr']['learning_rate']} ε={cfg['sr']['varepsilon']} | {run_dir.name}"),
    }

run_dirs = [d for folder in RESULTS_DIRS for d in folder.glob("N*_*") if (d / "results.npz").exists()]
all_runs = sorted((load_run(d) for d in run_dirs), key=lambda r: r["time"])

print(f"{len(all_runs)} runs found in {[str(f) for f in RESULTS_DIRS]}\n")
print(f"{'#':>3}  {'run':<30}{'N':>4}{'d':>3}{'smp':>5}{'δ':>6}{'η':>7}{'ε':>7}{'iters':>7}{'E_final':>12}  status")
for i, r in enumerate(all_runs):
    s, c = r["system"], r["cfg"]
    status = "DIVERGED" if r["diverged"] else ("converged" if r["iter_tol"] else f"rel err {r['rel_err'][-1]:.1e}")
    print(f"{i:>3}  {r['name']:<30}{s.n_particles:>4}{s.dim:>3}{SAMPLER_TAGS[r['sampler_type']]:>5}{r['sampler_cfg']['step_size']:>6}{c['sr']['learning_rate']:>7}"
          f"{c['sr']['varepsilon']:>7}{len(r['energy']):>7}{r['E_final']:>12.5g}  {status}")

## 1b. Choose the runs to analyse

Every filter left as `None` is ignored; the ones that are set are combined (a run must pass all of them). The result is `runs`, which is what the rest of the notebook uses.

| Filter | Example | Keeps |
|---|---|---|
| `SELECT_INDEX` | `[0, 4]` | runs by their `#` in the list above |
| `SELECT_NAMES` | `["N4_20261006-121907"]` | runs by folder name |
| `SELECT_N`, `SELECT_DIM` | `[40]`, `[2]` | runs with those N / dimensions |
| `SELECT_SAMPLER` | `["gibbs"]` | runs trained with those samplers (`"metropolis"`, `"gibbs"`) |
| `SELECT_WHERE` | `lambda r: r["cfg"]["sr"]["learning_rate"] >= 0.01` | any condition on a run (its `cfg`, `system`, history...) |
| `SELECT_LAST` | `3` | only the most recent ones, after the other filters |
| `SKIP_DIVERGED` | `True` | drops runs with NaN/inf |

In [ ]:
SELECT_INDEX  = None
SELECT_NAMES  = None
SELECT_N      = None
SELECT_DIM    = None
SELECT_SAMPLER = None   #e.g. ["gibbs"] or ["metropolis"]
SELECT_WHERE  = None
SELECT_LAST   = 2
SKIP_DIVERGED = True

def selected(i, r):
    return ((SELECT_INDEX is None or i in SELECT_INDEX)
            and (SELECT_NAMES is None or r["name"] in SELECT_NAMES)
            and (SELECT_N is None or r["system"].n_particles in SELECT_N)
            and (SELECT_DIM is None or r["system"].dim in SELECT_DIM)
            and (SELECT_SAMPLER is None or r["sampler_type"] in SELECT_SAMPLER)
            and (SELECT_WHERE is None or SELECT_WHERE(r))
            and not (SKIP_DIVERGED and r["diverged"]))

runs = [r for i, r in enumerate(all_runs) if selected(i, r)]
if SELECT_LAST is not None:
    runs = runs[-SELECT_LAST:]

#A name in SELECT_NAMES that matches nothing is usually a typo, so we say so
if SELECT_NAMES is not None:
    missing = set(SELECT_NAMES) - {r["name"] for r in all_runs}
    if missing:
        print("Warning: not found:", sorted(missing))
if not runs:
    raise ValueError("No run passes the filters: relax them (see the list above)")

print(f"{len(runs)} of {len(all_runs)} runs selected:")
for r in runs:
    print("  ", r["label"])

In [ ]:
#Summary table, one row per run
header = f"{'run':<30}{'N':>3}{'d':>3}{'smp':>5}{'ω':>5}{'iters':>7}{'E_final':>20}{'E_0':>8}{'rel err':>10}{'Var(E)':>10}{'acc':>6}{'α':>8}{'it<tol':>8}"
print(header)
print("-" * len(header))
for r in runs:
    s = r["system"]
    E = f"{r['E_final']:.6f}±{r['E_err']:.6f}"
    print(f"{r['name']:<30}{s.n_particles:>3}{s.dim:>3}{SAMPLER_TAGS[r['sampler_type']]:>5}{s.omega:>5.1f}{len(r['energy']):>7}{E:>20}{s.exact_energy:>8.3f}"
          f"{r['rel_err'][-1]:>10.1e}{r['variance'][-1]:>10.1e}{r['acc_last']:>6.2f}{r['alpha'][-1]:>8.4f}{str(r['iter_tol']):>8}")

## 2. Training curves

- $E/E_0$ is used instead of $E$, so systems with different $N$ or $d$ share an axis.
- $\mathrm{Var}(E_{loc})$ is the cleanest convergence signal: it is exactly 0 for an eigenstate, whatever the sampling noise.
- The green band is the acceptance range `train.py` warns about at the end of training. For Gibbs runs the acceptance is per particle, so it is not directly comparable with the Metropolis one.
- $\alpha$ does not need to reach $\omega/2$: the RBM can provide part of the Gaussian decay.
- The last panel plots the relative error against the variance at every iteration. Near the ground state both go to 0 together, so a run with a low variance but a high error would point to a bias.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8.5))
(ax_E, ax_rel, ax_var), (ax_acc, ax_alpha, ax_scatter) = axes

for r in runs:
    it = np.arange(1, len(r["energy"]) + 1)
    line, = ax_E.plot(it, r["energy"] / r["system"].exact_energy, label=r["label"])
    c = line.get_color()
    ax_rel.semilogy(it, r["rel_err"], color=c)
    ax_var.semilogy(it, r["variance"], color=c)
    ax_acc.plot(it, r["acceptance"], color=c)
    ax_alpha.plot(it, r["alpha"], color=c)
    ax_scatter.loglog(r["variance"], r["rel_err"], ".", color=c, ms=3)
    ax_alpha.axhline(r["system"].omega / 2, color=c, ls=":", lw=1)   #pure-Gaussian value ω/2

ax_E.axhline(1, color="k", ls="--", lw=1)
ax_rel.axhline(REL_TOL, color="k", ls="--", lw=1)
ax_acc.axhspan(ACC_MIN, ACC_MAX, color="green", alpha=0.1)

ax_E.set(title="E / E_0", xlabel="SR iteration")
ax_rel.set(title="|E − E_0| / E_0", xlabel="SR iteration")
ax_var.set(title="Var(E_loc)", xlabel="SR iteration")
ax_acc.set(title="acceptance (per particle for Gibbs)", xlabel="SR iteration")
ax_alpha.set(title="α (dotted: ω/2)", xlabel="SR iteration")
ax_scatter.set(title="Relative error vs Var(E_loc)", xlabel="Var(E_loc)", ylabel="|E − E_0| / E_0")
ax_E.legend(fontsize=7)
fig.tight_layout()
plt.show()

## 3. Does the acceptance drift matter?

The acceptance is checked once, after thermalization, but it drifts while $\psi$ narrows. The table and the plot compare how much each run drifted with how fast it converged.

A low acceptance does not bias the energy, but it makes consecutive samples more correlated. If runs that end with a low acceptance also converge more slowly (more iterations to reach `REL_TOL`) or end with a higher variance, it is worth monitoring the acceptance in `train.py` or lowering `step_size`.

In [ ]:
print(f"{'run':<28}{'acc first':>10}{'acc last':>10}{'min':>7}{'iters outside':>15}{'it<tol':>8}{'Var(E) last':>13}")
for r in runs:
    acc = r["acceptance"]
    outside = int(np.sum((acc < ACC_MIN) | (acc > ACC_MAX)))
    print(f"{r['name']:<28}{acc[0]:>10.2f}{r['acc_last']:>10.2f}{acc.min():>7.2f}{outside:>15}{str(r['iter_tol']):>8}{r['variance'][-1]:>13.1e}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for r in runs:
    ax1.scatter(r["acc_last"], r["iter_tol"] if r["iter_tol"] else np.nan, label=r["label"])
    ax2.scatter(r["acc_last"], r["variance"][-1])
for ax in (ax1, ax2):
    ax.axvspan(ACC_MIN, ACC_MAX, color="green", alpha=0.1)
    ax.set_xlabel("final acceptance (mean of the last 10%)")
ax1.set_ylabel(f"iterations to reach rel. error < {REL_TOL:g}")
ax2.set(ylabel="final Var(E_loc)", yscale="log")
ax1.legend(fontsize=7)
fig.tight_layout()
plt.show()

## 4. The trained wavefunction

Every selected run is analysed (diverged runs are skipped). The model of each run is rebuilt from its own config, with `dim` taken from `[system]` as `train.py` does. `build_sampler` rebuilds the sampler the run was trained with, so sections 5 and 6 re-sample with Metropolis or Gibbs accordingly.

The exact ground state is $\log\psi_0 = -\tfrac{\omega}{2}\sum_i |x_i|^2 + \text{const}$. Moving particle 0 along the $x$ axis with the others at the origin, the trained $\log|\psi|$ must follow the parabola $-\tfrac{\omega}{2}x^2$. It is shifted to 0 at $x = 0$ to remove the normalization. The phase is checked in section 6, on the configurations that $|\psi|^2$ actually samples rather than on a single line.

In [ ]:
def build_model(r):
    #Rebuilds the trained model and its parameters from the run's own config, with dim taken from [system]
    #as train.py does
    if r["diverged"]:
        raise ValueError(f"{r['name']} diverged (NaN/inf): its wavefunction cannot be analysed")
    model = NQS(**r["cfg"]["network"], dim=r["system"].dim)
    _, unravel = flatten_params(model.init(jax.random.PRNGKey(0)))   #any key: only the structure is used
    return model, unravel(jnp.asarray(r["theta"]))

def build_log_psi(r):
    model, params = build_model(r)
    return jax.jit(lambda x: model.apply(params, x))

def build_sampler(r):
    #The sampler the run was trained with, and sample(walkers, n, key) with the same signature for both:
    #Metropolis needs log_psi(x), Gibbs needs the parameters dictionary (as sample_fn in train.py)
    model, params = build_model(r)
    sampler = SAMPLERS[r["sampler_type"]](**r["sampler_cfg"])
    if r["sampler_type"] == "gibbs":
        sample = lambda w, n, k: sampler.sample(params, w, n, k)
    else:
        sample = lambda w, n, k: sampler.sample(lambda x: model.apply(params, x), w, n, k)
    return sampler, sample

#Cut: particle 0 at (s, 0, ...), the others at the origin
s = np.linspace(-3, 3, 201)
i0 = len(s) // 2   #index of s = 0
inside = np.abs(s) < 2   #|psi|^2 weights the region |x| < ~2, where the comparison matters

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
print(f"{'run':<28}{'N':>4}{'d':>3}{'max |log|ψ| − log|ψ_0|| (|x|<2)':>34}")
for r in runs:
    if r["diverged"]:
        print(f"{r['name']:<28}  DIVERGED, skipped")
        continue
    sys_r = r["system"]
    X = np.zeros((len(s), sys_r.n_particles, sys_r.dim))
    X[:, 0, 0] = s
    f = np.asarray(jax.vmap(build_log_psi(r))(jnp.asarray(X)))
    log_amp = f.real - f.real[i0]
    exact = -0.5 * sys_r.omega * s**2
    line, = ax1.plot(s, log_amp, label=r["label"])
    ax2.plot(s, log_amp - exact, color=line.get_color())
    print(f"{r['name']:<28}{sys_r.n_particles:>4}{sys_r.dim:>3}{np.abs(log_amp - exact)[inside].max():>34.2e}")

#Exact parabola, once per distinct ω among the selected runs
for omega in sorted({r["system"].omega for r in runs}):
    ax1.plot(s, -0.5 * omega * s**2, "k--", lw=1, label=f"exact −ω x²/2 (ω={omega:g})")
ax1.set(title="log|ψ| (shifted to 0 at x = 0)", xlabel="x of particle 0")
ax1.legend(fontsize=7)
ax2.axhline(0, color="k", ls="--", lw=1)
ax2.set(title="log|ψ| − log|ψ_0|", xlabel="x of particle 0")
fig.tight_layout()
plt.show()

## 5. Final evaluation at fixed θ

This section and the first part of section 6 analyse one run, chosen in `RUN_NAME` (by default `None`, the most recent of the selected runs). It must be one of `runs`, and not diverged.

Sampling uses the same sampler (and settings) the run was trained with.

The energy printed by `train.py` averages the last iterations while $\theta$ is still changing, and its error bar ignores correlations. Here $\theta$ is fixed and many more samples are drawn:

- The error bar uses the mean of each chain. Chains are independent, so the spread of the chain means gives an honest error even though consecutive samples of one chain are correlated.
- $\mathrm{Var}(E_{loc})$ is split into real and imaginary parts. The variance stored by `train.py` is that of the complex $E_{loc}$, so it includes the imaginary part, which comes from the learned phase not being exactly constant.
- The one-body density is compared with the exact one: in $\psi_0$ every coordinate is Gaussian with variance $1/(2\omega)$, so $\langle |x_i|^2\rangle = d/(2\omega)$.

`N_EVAL` sets the number of recorded samples per chain. Raise it for a smaller error bar.

In [ ]:
RUN_NAME = None   #e.g. "N4_20261006-121907"; None = most recent selected run

if RUN_NAME is None:
    RUN = runs[-1]
else:
    matches = [r for r in runs if r["name"] == RUN_NAME]
    if not matches:
        raise ValueError(f"{RUN_NAME} is not among the selected runs: add it to the filters in section 1b")
    RUN = matches[0]

cfg, system = RUN["cfg"], RUN["system"]
log_psi_x = build_log_psi(RUN)   #raises if the run diverged
print("Analysing", RUN["label"])

In [ ]:
N_THERM, N_EVAL = 200, 200   #sweeps: discarded / recorded per chain

sampler, sample = build_sampler(RUN)
key_w, key_t, key_s = jax.random.split(jax.random.PRNGKey(1), 3)
walkers = sampler.init_walkers(key_w, system.n_particles, system.dim)

#Thermalize from scratch, then record N_EVAL sweeps at fixed theta
_, walkers, acc_therm = sample(walkers, N_THERM, key_t)
samples, walkers, acc_eval = sample(walkers, N_EVAL, key_s)

#E_loc needs a (N*dim)^2 Hessian per sample: all samples at once do not fit in GPU memory for large N,
#so we evaluate them in batches of the size of one training iteration
E_LOC_BATCH = sampler.n_chains * cfg["training"]["n_samples"]
batch_E = jax.jit(lambda x: system.batch_local_energy(log_psi_x, x))
E_loc_c = np.concatenate([np.asarray(batch_E(samples[i:i + E_LOC_BATCH]))
                          for i in range(0, samples.shape[0], E_LOC_BATCH)])
E_loc_c = E_loc_c.reshape(N_EVAL, sampler.n_chains)   #samples are sweep-major
E_loc = E_loc_c.real   #<H> is real: the imaginary part of E_loc averages to 0

chain_means = E_loc.mean(axis=0)
E_eval = chain_means.mean()
E_eval_err = chain_means.std(ddof=1) / np.sqrt(sampler.n_chains)
print(f"{N_EVAL * sampler.n_chains} samples ({RUN['sampler_type']}), acceptance = {float(acc_eval):.2f}")
print(f"E = {E_eval:.6f} ± {E_eval_err:.6f}   (E_0 = {system.exact_energy:.6f}, "
      f"deviation = {(E_eval - system.exact_energy) / E_eval_err:+.1f} σ)")
#SR.step reports jnp.var of the complex E_loc, which also counts the imaginary part. It is not 0
#because the learned phase is not exactly constant, so it is reported separately
print(f"Var(E_loc) = {E_loc_c.var():.2e}  (real part {E_loc_c.real.var():.2e}, imaginary part {E_loc_c.imag.var():.2e})")
print(f"train.py estimate: E = {RUN['E_final']:.6f} ± {RUN['E_err']:.6f}")

In [ ]:
#One-body density: every coordinate of every particle, against the exact Gaussian of variance 1/(2ω)
coords = np.asarray(samples).reshape(-1)
grid = np.linspace(-3, 3, 300)
var0 = 1 / (2 * system.omega)

r2 = np.sum(np.asarray(samples) ** 2, axis=2)   #|x_i|^2 per sample and particle
print(f"<|x_i|^2> = {r2.mean():.4f}   (exact d/(2ω) = {system.dim * var0:.4f})")

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(coords, bins=120, density=True, alpha=0.6, label="NQS samples")
ax.plot(grid, np.exp(-grid**2 / (2 * var0)) / np.sqrt(2 * np.pi * var0), "k--", label="exact")
ax.set(title="One-body density (one coordinate)", xlabel="x")
ax.legend()
plt.show()

## 6. Is the phase constant?

The bosonic ground state is real and positive, so the phase $\varphi = \mathrm{Im}\log\psi$ must take the same value on every configuration (a global phase is irrelevant). A phase that is not constant costs energy: the real part of $E_{loc}$ contains $+\tfrac12|\nabla\varphi|^2$, so

$$E - E_0 \;\supseteq\; \tfrac12\langle|\nabla\varphi|^2\rangle_{|\psi|^2}.$$

- For the run of section 5: the distribution of $\varphi - \langle\varphi\rangle$ over the samples of section 5. A constant phase gives a single spike at 0.
- For every selected run: the spread of $\varphi$ and its energy cost, compared with the total error $E - E_0$. A phase that is exactly constant shows **0** (e.g. when the output weights `u` of the phase network start at 0).
- **During training:** `train.py` saves `phase_std`, the std of $\varphi$ over the samples of every iteration, so the last plot shows how the phase becomes constant. Runs saved before this was added do not have it and are skipped.

In [ ]:
def batched(f, x, size):
    #Evaluates f on x in batches, so large N fits in GPU memory (same reason as E_loc in section 5)
    return np.concatenate([np.asarray(f(x[i:i + size])) for i in range(0, x.shape[0], size)])

def phase_stats(log_psi_x, samples, size):
    phi = batched(jax.jit(jax.vmap(lambda x: jnp.imag(log_psi_x(x)))), samples, size)
    grad_phi = jax.jit(jax.vmap(lambda x: jax.grad(lambda y: jnp.imag(log_psi_x(y)))(x)))
    g2 = batched(lambda s: jnp.sum(grad_phi(s) ** 2, axis=(1, 2)), samples, size)
    return phi - phi.mean(), 0.5 * g2.mean()   #centered phase (global phase removed), energy cost

phi, phase_cost = phase_stats(log_psi_x, samples, E_LOC_BATCH)
print(f"{RUN['name']}: std(φ) = {phi.std():.2e}, max − min = {np.ptp(phi):.2e}")
print(f"energy cost of the phase ½<|∇φ|²> = {phase_cost:.2e}   vs   E − E_0 = {E_eval - system.exact_energy:.2e}")

fig, ax = plt.subplots(figsize=(6, 4))
if np.ptp(phi) > 0:
    ax.hist(phi, bins=100, density=True, alpha=0.7)
else:
    ax.axvline(0, lw=3)   #exactly constant: a spike at 0
ax.set(title=f"φ − <φ> over |ψ|² samples  ({RUN['name']})", xlabel="φ − <φ>")
plt.show()

In [ ]:
#Same check on every selected run, with a shorter sampling (the spread of phi needs fewer samples than E)
N_THERM_PHASE, N_PHASE = 200, 20

rows = []
for r in runs:
    c, s_r = r["cfg"], r["system"]
    lp = build_log_psi(r)
    smp, sample_r = build_sampler(r)   #the sampler the run was trained with
    k1, k2, k3 = jax.random.split(jax.random.PRNGKey(2), 3)
    w = smp.init_walkers(k1, s_r.n_particles, s_r.dim)
    _, w, _ = sample_r(w, N_THERM_PHASE, k2)
    x, _, _ = sample_r(w, N_PHASE, k3)
    ph, cost = phase_stats(lp, x, smp.n_chains * c["training"]["n_samples"])
    rows.append((r, ph.std(), cost, r["E_final"] - s_r.exact_energy))

print(f"{'run':<30}{'N':>4}{'std(φ)':>11}{'½<|∇φ|²>':>11}{'E − E_0':>11}{'phase share':>15}")
for r, sd, cost, dE in rows:
    #The share is only meaningful when E − E_0 is clearly above its own error bar
    share = f"{100 * cost / dE:.0f}%" if dE > 3 * r["E_err"] else "within error"
    print(f"{r['name']:<30}{r['system'].n_particles:>4}{sd:>11.2e}{cost:>11.2e}{dE:>11.2e}{share:>15}")

fig, ax = plt.subplots(figsize=(8, 0.5 * len(rows) + 1.5))
names = [r["name"] for r, *_ in rows]
costs = [max(cost, 1e-12) for _, _, cost, _ in rows]   #log axis: exact zeros are drawn at 1e-12
ax.barh(names, costs)
for i, (_, _, cost, _) in enumerate(rows):
    if cost == 0:
        ax.text(1e-12, i, "  exactly 0", va="center")
ax.set(xscale="log", xlabel="energy cost of the phase ½<|∇φ|²>", title="Is the phase constant?")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
#Phase during training: std of phi over the samples of every SR iteration (saved by train.py)
FLOOR = 1e-12   #log axis: a phase that is exactly constant (std = 0) is drawn at FLOOR

with_phase = [r for r in runs if r["phase_std"] is not None]
skipped = [r["name"] for r in runs if r["phase_std"] is None]
if skipped:
    print("No phase_std saved (older runs):", skipped)

if with_phase:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for r in with_phase:
        ps = r["phase_std"]
        label = r["label"] + ("  (exactly 0)" if np.all(ps == 0) else "")
        ax.semilogy(np.arange(1, len(ps) + 1), np.maximum(ps, FLOOR), label=label)
    ax.set(title="std(φ) over the samples of each iteration", xlabel="SR iteration", ylabel="std(φ)")
    ax.legend(fontsize=7)
    plt.tight_layout()
    plt.show()
    for r in with_phase:
        ps = r["phase_std"]
        print(f"{r['name']:<28} std(φ): first {ps[0]:.2e}  last {ps[-1]:.2e}")